# AeroVision LK — SAHI Inference

**Week 2 | Day 8–10**  
**Model:** YOLOv8s baseline + SAHI sliced inference  
**Dataset:** VisDrone2019-DET val split (548 images)

---
**Baseline results (notebook 02):**
- mAP@0.5: **43.57%** — mAP@0.5:0.95: **25.89%**
- Weak spots: bicycle 13.3% · people 34.0% · three_wheeler 35.5%

**Why SAHI?**  
78.7% of VisDrone objects are < 50×50 px. YOLOv8 letterboxes every image to 640×640,  
shrinking those objects to ~33 px or less. SAHI tiles the full-resolution image into  
overlapping 512×512 slices, runs detection on each slice, and merges results — keeping  
small objects large enough for the detector to see.

**Updated target:** 55%+ mAP@0.5 (+11pp over baseline)

---

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import datetime
import yaml
import time
import mlflow
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# ── Project root ──────────────────────────────────────────────────────────────
def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'VisDrone_Dataset').exists():
            return candidate
    raise RuntimeError('Cannot find project root. Run from 01_aerovision_lk/ or research/')

PROJECT_ROOT      = find_project_root()
DATASET_ROOT      = PROJECT_ROOT / 'data' / 'VisDrone_Dataset'
DATASET_YAML_ABS  = DATASET_ROOT / 'visdrone_abs.yaml'
BASELINE_WEIGHTS  = PROJECT_ROOT / 'weights' / 'yolov8s_baseline.pt'
FIGURES_DIR       = PROJECT_ROOT / 'reports' / 'figures'
ANALYSIS_DIR      = PROJECT_ROOT / 'analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

VAL_IMAGES_DIR = DATASET_ROOT / 'VisDrone2019-DET-val' / 'images'
VAL_LABELS_DIR = DATASET_ROOT / 'VisDrone2019-DET-val' / 'labels'

CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van',
               'truck', 'three_wheeler', 'bus', 'motor']
NC = len(CLASS_NAMES)

COLORS_RGB = [
    (255,  59,  59),
    (255, 149,   0),
    (255, 214,   0),
    ( 52, 199,  89),
    (  0, 190, 235),
    (  0, 122, 255),
    (175,  82, 222),
    (255,  45, 143),
    (162, 132,  94),
]
COLORS_BGR = [(b, g, r) for r, g, b in COLORS_RGB]

# ── Load baseline metrics ─────────────────────────────────────────────────────
baseline_csv = ANALYSIS_DIR / 'baseline_metrics.csv'
baseline_row = pd.read_csv(baseline_csv).iloc[-1]
BASELINE_MAP50   = float(baseline_row['mAP50'])
BASELINE_MAP5095 = float(baseline_row['mAP50_95'])

assert DATASET_YAML_ABS.exists(), f'visdrone_abs.yaml not found. Run notebook 02 first.'
assert BASELINE_WEIGHTS.exists(), f'baseline weights not found at {BASELINE_WEIGHTS}'

print(f'Project root     : {PROJECT_ROOT}')
print(f'Baseline weights : {BASELINE_WEIGHTS}')
print(f'Val images       : {len(list(VAL_IMAGES_DIR.glob("*.jpg")))} images')
print(f'Baseline mAP@0.5 : {BASELINE_MAP50*100:.2f}%')
print(f'Target mAP@0.5   : 55%+ (+{(0.55 - BASELINE_MAP50)*100:.1f}pp)')

---
## 1. Load Models

Load baseline weights into both raw YOLO (for baseline inference) and SAHI `AutoDetectionModel`.

In [ ]:
# ── Raw YOLO model (baseline inference) ──────────────────────────────────────
yolo_model = YOLO(BASELINE_WEIGHTS)
print(f'YOLO model loaded : {BASELINE_WEIGHTS.name}')

# ── SAHI AutoDetectionModel ───────────────────────────────────────────────────
detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=str(BASELINE_WEIGHTS),
    confidence_threshold=0.25,
    device='cuda:0',
)
print(f'SAHI model loaded : confidence_threshold=0.25, device=cuda:0')

# ── Default SAHI config ───────────────────────────────────────────────────────
SLICE_SIZE    = 512
OVERLAP_RATIO = 0.2
print(f'Default SAHI config: slice={SLICE_SIZE}, overlap={OVERLAP_RATIO}')

---
## 2. Shared Helpers

In [ ]:
def load_label(label_path: Path):
    """Return (boxes_norm [N,4], class_ids [N]) from a YOLO label file."""
    rows, cids = [], []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            cids.append(int(parts[0]))
            rows.append([float(x) for x in parts[1:]])
    boxes = np.array(rows, dtype=np.float32) if rows else np.zeros((0, 4), dtype=np.float32)
    return boxes, cids


def yolo_to_xyxy(boxes_norm, img_w, img_h):
    """Convert YOLO normalised [cx,cy,w,h] -> pixel [x1,y1,x2,y2]."""
    if len(boxes_norm) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    cx, cy, bw, bh = boxes_norm[:,0], boxes_norm[:,1], boxes_norm[:,2], boxes_norm[:,3]
    x1 = (cx - bw/2) * img_w
    y1 = (cy - bh/2) * img_h
    x2 = (cx + bw/2) * img_w
    y2 = (cy + bh/2) * img_h
    return np.stack([x1, y1, x2, y2], axis=1).astype(np.float32)


def iou_matrix(gt_xyxy, pred_xyxy):
    """Vectorized IoU [N_gt, N_pred]. No external dependencies."""
    if len(gt_xyxy) == 0 or len(pred_xyxy) == 0:
        return np.zeros((len(gt_xyxy), len(pred_xyxy)), dtype=np.float32)
    gt   = gt_xyxy[:, None, :]
    pred = pred_xyxy[None, :, :]
    ix1  = np.maximum(gt[...,0], pred[...,0])
    iy1  = np.maximum(gt[...,1], pred[...,1])
    ix2  = np.minimum(gt[...,2], pred[...,2])
    iy2  = np.minimum(gt[...,3], pred[...,3])
    iw   = np.maximum(0.0, ix2 - ix1)
    ih   = np.maximum(0.0, iy2 - iy1)
    inter = iw * ih
    area_gt   = (gt_xyxy[:,2]-gt_xyxy[:,0]) * (gt_xyxy[:,3]-gt_xyxy[:,1])
    area_pred = (pred_xyxy[:,2]-pred_xyxy[:,0]) * (pred_xyxy[:,3]-pred_xyxy[:,1])
    union = area_gt[:,None] + area_pred[None,:] - inter
    return inter / np.maximum(union, 1e-6)


def compute_ap(recalls, precisions):
    """Compute AP using 101-point interpolation (COCO-style)."""
    recalls    = np.concatenate(([0.0], recalls,    [1.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))
    # Make precision monotonically decreasing
    for i in range(len(precisions)-2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i+1])
    thresholds = np.linspace(0, 1, 101)
    ap = 0.0
    for t in thresholds:
        mask = recalls >= t
        ap  += precisions[mask].max() if mask.any() else 0.0
    return ap / 101.0


def compute_map(all_preds, all_gts, iou_threshold=0.5):
    """
    Compute mAP across all classes.
    all_preds: list of dicts {class_id, conf, x1,y1,x2,y2, img_id}
    all_gts:   list of dicts {class_id, x1,y1,x2,y2, img_id, matched}
    Returns: mAP (float), per_class_ap dict
    """
    per_class_ap = {}
    for cid in range(NC):
        preds = [p for p in all_preds if p['class_id'] == cid]
        gts   = [g for g in all_gts   if g['class_id'] == cid]
        if len(gts) == 0:
            continue
        # Sort predictions by descending confidence
        preds = sorted(preds, key=lambda x: -x['conf'])
        matched_gt = set()
        tp = np.zeros(len(preds))
        fp = np.zeros(len(preds))
        for i, pred in enumerate(preds):
            img_gts = [g for g in gts if g['img_id'] == pred['img_id']]
            if len(img_gts) == 0:
                fp[i] = 1
                continue
            pred_box = np.array([[pred['x1'], pred['y1'], pred['x2'], pred['y2']]], dtype=np.float32)
            gt_boxes = np.array([[g['x1'],g['y1'],g['x2'],g['y2']] for g in img_gts], dtype=np.float32)
            ious     = iou_matrix(gt_boxes, pred_box)[:,0]  # [N_gt]
            best_idx = int(np.argmax(ious))
            gt_key   = (pred['img_id'], img_gts[best_idx]['gt_idx'])
            if ious[best_idx] >= iou_threshold and gt_key not in matched_gt:
                tp[i] = 1
                matched_gt.add(gt_key)
            else:
                fp[i] = 1
        cum_tp  = np.cumsum(tp)
        cum_fp  = np.cumsum(fp)
        recalls    = cum_tp / (len(gts) + 1e-6)
        precisions = cum_tp / (cum_tp + cum_fp + 1e-6)
        per_class_ap[cid] = compute_ap(recalls, precisions)
    mAP = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return mAP, per_class_ap


def get_most_crowded(labels_dir: Path, n=5):
    files  = list(labels_dir.glob('*.txt'))
    counts = [(f, sum(1 for l in f.read_text().splitlines() if l.strip())) for f in files]
    return sorted(counts, key=lambda x: x[1], reverse=True)[:n]


def sahi_to_xyxy(sahi_result, img_w, img_h):
    """Extract (boxes_xyxy, class_ids, confs) from a SAHI SlicedPrediction."""
    boxes, cids, confs = [], [], []
    for obj in sahi_result.object_prediction_list:
        b = obj.bbox
        boxes.append([b.minx, b.miny, b.maxx, b.maxy])
        cids.append(int(obj.category.id))
        confs.append(float(obj.score.value))
    boxes = np.array(boxes,  dtype=np.float32) if boxes else np.zeros((0,4), dtype=np.float32)
    return boxes, cids, confs


print('Helpers defined.')

---
## 3. Quick Visual Comparison

The 2 most crowded val scenes — baseline 640px vs SAHI 512/0.2.  
This is the key qualitative proof that SAHI recovers small objects.

In [ ]:
crowded = get_most_crowded(VAL_LABELS_DIR, n=2)

fig, axes = plt.subplots(2, 2, figsize=(22, 12))
fig.suptitle(
    'Baseline YOLOv8s @ 640px  vs  SAHI (512×512 slices, overlap 0.2)\n'
    'Left: Baseline | Right: SAHI — more boxes = more small objects detected',
    fontsize=13, fontweight='bold'
)

for row_idx, (label_path, ann_count) in enumerate(crowded):
    img_path = VAL_IMAGES_DIR / (label_path.stem + '.jpg')
    img_bgr  = cv2.imread(str(img_path))
    h, w     = img_bgr.shape[:2]

    # ── Baseline inference ────────────────────────────────────────────────────
    base_result  = yolo_model.predict(source=str(img_path), imgsz=640,
                                      conf=0.25, iou=0.45, verbose=False)[0]
    base_boxes   = base_result.boxes.xyxy.cpu().numpy()  if len(base_result.boxes) else np.zeros((0,4))
    base_cids    = base_result.boxes.cls.cpu().numpy().astype(int) if len(base_result.boxes) else []

    base_canvas  = img_bgr.copy()
    for (x1,y1,x2,y2), cid in zip(base_boxes.astype(int), base_cids):
        cv2.rectangle(base_canvas, (x1,y1), (x2,y2), COLORS_BGR[cid % NC], 1)

    # ── SAHI inference ────────────────────────────────────────────────────────
    sahi_result  = get_sliced_prediction(
        str(img_path), detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=False,
    )
    sahi_boxes, sahi_cids, _ = sahi_to_xyxy(sahi_result, w, h)

    sahi_canvas  = img_bgr.copy()
    for (x1,y1,x2,y2), cid in zip(sahi_boxes.astype(int), sahi_cids):
        cv2.rectangle(sahi_canvas, (x1,y1), (x2,y2), COLORS_BGR[cid % NC], 1)

    # ── Plot ──────────────────────────────────────────────────────────────────
    ax_b = axes[row_idx, 0]
    ax_s = axes[row_idx, 1]

    ax_b.imshow(cv2.cvtColor(base_canvas, cv2.COLOR_BGR2RGB))
    ax_b.set_title(f'Baseline: {len(base_boxes)} detections  |  GT: {ann_count}', fontsize=10)
    ax_b.axis('off')

    ax_s.imshow(cv2.cvtColor(sahi_canvas, cv2.COLOR_BGR2RGB))
    ax_s.set_title(f'SAHI (512/0.2): {len(sahi_boxes)} detections  |  GT: {ann_count}', fontsize=10,
                   color='darkgreen' if len(sahi_boxes) > len(base_boxes) else 'black')
    ax_s.axis('off')

legend_patches = [mpatches.Patch(color=[c/255 for c in col], label=name)
                  for name, col in zip(CLASS_NAMES, COLORS_RGB)]
fig.legend(handles=legend_patches, loc='lower center', ncol=9,
           fontsize=8.5, frameon=True, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
out_path = FIGURES_DIR / 'sahi_visual_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

---
## 4. Full Val Set Evaluation — SAHI

Run SAHI inference on all 548 val images and compute mAP@0.5 and mAP@0.5:0.95  
using the same vectorized IoU + precision-recall curve approach as notebook 02.  
Expected runtime: ~20 min on RTX 3050.

In [ ]:
val_images = sorted(VAL_IMAGES_DIR.glob('*.jpg'))
n_images   = len(val_images)
t_start    = datetime.datetime.now()

print('=' * 58)
print('  SAHI Full Val Evaluation')
print('=' * 58)
print(f'  Val image count  : {n_images}')
print(f'  Slice config     : {SLICE_SIZE}×{SLICE_SIZE}, overlap {OVERLAP_RATIO}')
print(f'  Conf threshold   : 0.25')
print(f'  Estimated runtime: ~20 min on RTX 3050')
print(f'  Start time       : {t_start.strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 58)

all_preds_50   = []   # for mAP@0.5
all_gts_50     = []   # for mAP@0.5
all_preds_5095 = []   # for mAP@0.5:0.95
all_gts_5095   = []   # for mAP@0.5:0.95

for img_idx, img_path in enumerate(val_images):
    label_path = VAL_LABELS_DIR / (img_path.stem + '.txt')
    if not label_path.exists():
        continue

    img       = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2]
    img_id    = img_path.stem

    # ── Ground truth ──────────────────────────────────────────────────────────
    gt_norm, gt_cids = load_label(label_path)
    gt_xyxy          = yolo_to_xyxy(gt_norm, img_w, img_h)

    for gt_idx, (box, cid) in enumerate(zip(gt_xyxy, gt_cids)):
        entry = {'class_id': cid, 'img_id': img_id, 'gt_idx': gt_idx,
                 'x1': box[0], 'y1': box[1], 'x2': box[2], 'y2': box[3]}
        all_gts_50.append(entry)
        all_gts_5095.append(entry.copy())

    # ── SAHI inference ────────────────────────────────────────────────────────
    sahi_result = get_sliced_prediction(
        str(img_path), detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=False,
    )
    pred_boxes, pred_cids, pred_confs = sahi_to_xyxy(sahi_result, img_w, img_h)

    for box, cid, conf in zip(pred_boxes, pred_cids, pred_confs):
        entry = {'class_id': int(cid), 'conf': float(conf), 'img_id': img_id,
                 'x1': box[0], 'y1': box[1], 'x2': box[2], 'y2': box[3]}
        all_preds_50.append(entry)
        all_preds_5095.append(entry.copy())

    if (img_idx + 1) % 50 == 0 or (img_idx + 1) == n_images:
        elapsed = (datetime.datetime.now() - t_start).total_seconds()
        rate    = (img_idx + 1) / elapsed
        eta_s   = (n_images - img_idx - 1) / max(rate, 1e-6)
        print(f'  [{img_idx+1:>3}/{n_images}]  elapsed {elapsed/60:.1f} min  '
              f'ETA {eta_s/60:.1f} min  ({rate:.1f} img/s)')

t_end   = datetime.datetime.now()
elapsed = (t_end - t_start).total_seconds()
print()
print(f'  End time    : {t_end.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Actual time : {elapsed/60:.1f} min ({elapsed:.0f} s)')
print(f'  Total preds : {len(all_preds_50):,}')
print(f'  Total GTs   : {len(all_gts_50):,}')

---
## 5. Compute mAP@0.5 and mAP@0.5:0.95

In [ ]:
print('Computing mAP@0.5 ...')
sahi_map50, sahi_ap50_per_class = compute_map(all_preds_50, all_gts_50, iou_threshold=0.5)
print(f'  mAP@0.5 = {sahi_map50*100:.2f}%')

# mAP@0.5:0.95 — average over IoU thresholds 0.5..0.95 step 0.05
print('Computing mAP@0.5:0.95 (10 thresholds) ...')
iou_thresholds = np.arange(0.5, 1.0, 0.05)
map5095_values = []
ap5095_per_class_accum = {cid: [] for cid in range(NC)}

for thresh in iou_thresholds:
    m, per_cls = compute_map(all_preds_5095, all_gts_5095, iou_threshold=round(thresh, 2))
    map5095_values.append(m)
    for cid, ap in per_cls.items():
        ap5095_per_class_accum[cid].append(ap)
    print(f'  IoU={thresh:.2f} → mAP={m*100:.2f}%')

sahi_map5095 = float(np.mean(map5095_values))
sahi_ap5095_per_class = {cid: float(np.mean(vals))
                         for cid, vals in ap5095_per_class_accum.items() if vals}

print()
print(f'  mAP@0.5:0.95 = {sahi_map5095*100:.2f}%')

---
## 6. Results Comparison Table

In [ ]:
# ── Per-class comparison ──────────────────────────────────────────────────────
baseline_ap50 = {
    name: float(baseline_row[f'AP50_{name}'])
    for name in CLASS_NAMES if f'AP50_{name}' in baseline_row
}

per_class_rows = []
for cid, name in enumerate(CLASS_NAMES):
    b_ap  = baseline_ap50.get(name, 0.0)
    s_ap  = sahi_ap50_per_class.get(cid, 0.0)
    delta = s_ap - b_ap
    per_class_rows.append({
        'class_id':        cid,
        'class_name':      name,
        'baseline_AP50':   round(b_ap,  4),
        'sahi_AP50':       round(s_ap,  4),
        'delta_AP50':      round(delta, 4),
        'sahi_AP5095':     round(sahi_ap5095_per_class.get(cid, 0.0), 4),
    })

per_class_df = pd.DataFrame(per_class_rows).sort_values('delta_AP50', ascending=False).reset_index(drop=True)

# ── Print comparison table ────────────────────────────────────────────────────
delta_map50   = sahi_map50   - BASELINE_MAP50
delta_map5095 = sahi_map5095 - BASELINE_MAP5095

print('=' * 68)
print('  Results Comparison — YOLOv8s Baseline vs SAHI (512/0.2)')
print('=' * 68)
print(f'  {"Metric":<20} {"Baseline":>10} {"SAHI":>10} {"Delta":>10}')
print('  ' + '-' * 54)
print(f'  {"mAP@0.5":<20} {BASELINE_MAP50*100:>9.2f}% {sahi_map50*100:>9.2f}%'
      f' {delta_map50*100:>+9.2f}pp')
print(f'  {"mAP@0.5:0.95":<20} {BASELINE_MAP5095*100:>9.2f}% {sahi_map5095*100:>9.2f}%'
      f' {delta_map5095*100:>+9.2f}pp')
print()
print(f'  {"Class":<20} {"Baseline":>10} {"SAHI":>10} {"Delta":>10}')
print('  ' + '-' * 54)
for _, row in per_class_df.iterrows():
    marker = ' <--' if row['class_name'] in ['bicycle','people','three_wheeler'] else ''
    print(f'  {row["class_name"]:<20} {row["baseline_AP50"]*100:>9.2f}% '
          f'{row["sahi_AP50"]*100:>9.2f}% {row["delta_AP50"]*100:>+9.2f}pp{marker}')
print('=' * 68)

target_met = sahi_map50 >= 0.55
print(f'\n  Target (55%+ mAP@0.5): {"MET" if target_met else "NOT MET"} '
      f'({sahi_map50*100:.2f}%)')

---
## 7. Save Metrics & Log to MLflow

In [ ]:
# ── Build metrics row (same schema as baseline_metrics.csv) ──────────────────
sahi_row = {
    'model':     'yolov8s',
    'imgsz':     640,
    'epochs':    50,
    'batch':     4,
    'sahi':      True,
    'slice_size':    SLICE_SIZE,
    'overlap_ratio': OVERLAP_RATIO,
    'mAP50':     round(sahi_map50,   4),
    'mAP50_95':  round(sahi_map5095, 4),
    'precision': None,
    'recall':    None,
    'weights':   str(BASELINE_WEIGHTS),
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
}
for _, row in per_class_df.iterrows():
    sahi_row[f'AP50_{row["class_name"]}']   = row['sahi_AP50']
    sahi_row[f'AP5095_{row["class_name"]}'] = row['sahi_AP5095']

# ── Save to CSV ───────────────────────────────────────────────────────────────
sahi_csv_path = ANALYSIS_DIR / 'sahi_metrics.csv'
new_row_df    = pd.DataFrame([sahi_row])
if sahi_csv_path.exists():
    combined = pd.concat([pd.read_csv(sahi_csv_path), new_row_df], ignore_index=True)
else:
    combined = new_row_df
combined.to_csv(sahi_csv_path, index=False)
print(f'Saved → {sahi_csv_path}')

# ── Log to MLflow ─────────────────────────────────────────────────────────────
mlflow.set_tracking_uri((PROJECT_ROOT / 'mlruns').as_uri())
mlflow.set_experiment('aerovision-lk')

with mlflow.start_run(run_name=f'yolov8s-sahi-{SLICE_SIZE}-{int(OVERLAP_RATIO*100):02d}') as run:
    mlflow.log_params({
        'model':         'yolov8s',
        'sahi':          True,
        'slice_size':    SLICE_SIZE,
        'overlap_ratio': OVERLAP_RATIO,
        'conf_threshold': 0.25,
        'dataset':       'VisDrone2019-DET-9class',
    })
    mlflow.log_metric('val/mAP50',    sahi_map50)
    mlflow.log_metric('val/mAP50-95', sahi_map5095)
    mlflow.log_metric('delta_mAP50',  delta_map50)
    for cid, name in enumerate(CLASS_NAMES):
        if cid in sahi_ap50_per_class:
            mlflow.log_metric(f'AP50_{name}', sahi_ap50_per_class[cid])
    mlflow.log_artifact(str(sahi_csv_path), artifact_path='metrics')
    SAHI_RUN_ID = run.info.run_id

print(f'MLflow run ID : {SAHI_RUN_ID}')

---
## 8. Per-class Comparison Chart

In [ ]:
df_plot = per_class_df.sort_values('sahi_AP50', ascending=True)

fig, ax = plt.subplots(figsize=(13, 6))
y       = np.arange(len(df_plot))
h       = 0.35

# Highlight weak-spot classes
weak = {'bicycle', 'people', 'three_wheeler'}
base_colors = ['#c0392b' if n in weak else '#7f8c8d' for n in df_plot['class_name']]
sahi_colors = ['#27ae60' if n in weak else '#2980b9' for n in df_plot['class_name']]

bars_b = ax.barh(y - h/2, df_plot['baseline_AP50'] * 100, h,
                 color=base_colors, alpha=0.85, label='Baseline')
bars_s = ax.barh(y + h/2, df_plot['sahi_AP50']     * 100, h,
                 color=sahi_colors, alpha=0.85, label='SAHI 512/0.2')

# Delta labels
for i, (_, row) in enumerate(df_plot.iterrows()):
    delta = row['delta_AP50'] * 100
    ax.text(row['sahi_AP50']*100 + 0.5, y[i] + h/2,
            f'{delta:+.1f}pp', va='center', fontsize=8.5,
            color='darkgreen' if delta > 0 else 'darkred')

ax.set_yticks(y)
ax.set_yticklabels(df_plot['class_name'])
ax.set_xlabel('AP@0.5 (%)')
ax.set_title('Per-class AP@0.5: Baseline vs SAHI  |  Red/Green = weak-spot classes',
             fontsize=11)
ax.axvline(0, color='black', lw=0.5)
ax.legend(loc='lower right')

# Overall mAP annotation
ax.text(0.98, 0.02,
        f'Overall mAP@0.5:\nBaseline {BASELINE_MAP50*100:.2f}% → SAHI {sahi_map50*100:.2f}%'
        f'  ({delta_map50*100:+.2f}pp)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

plt.tight_layout()
out_path = FIGURES_DIR / 'sahi_per_class_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

---
## 9. Small Object Recall by Size Bucket

Split GT boxes into pixel-size buckets and measure recall for baseline vs SAHI.  
This directly proves SAHI's value on the small-object problem.

In [ ]:
SIZE_BUCKETS = [
    ('<32px',   0,    32),
    ('32-64px', 32,   64),
    ('64-128px',64,  128),
    ('>128px',  128, 9999),
]
IOU_THRESH = 0.5

# ── Collect per-image baseline predictions ────────────────────────────────────
print('Running baseline inference on val set for recall comparison...')
base_preds_by_img = {}
for img_path in val_images:
    result = yolo_model.predict(source=str(img_path), imgsz=640,
                                conf=0.25, iou=0.45, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy() if len(result.boxes) else np.zeros((0,4))
    base_preds_by_img[img_path.stem] = boxes
    if (list(val_images).index(img_path)+1) % 100 == 0:
        print(f'  {list(val_images).index(img_path)+1}/{len(val_images)}')

# ── Collect per-image SAHI predictions (already in all_preds_50) ──────────────
sahi_preds_by_img = {}
for pred in all_preds_50:
    img_id = pred['img_id']
    if img_id not in sahi_preds_by_img:
        sahi_preds_by_img[img_id] = []
    sahi_preds_by_img[img_id].append(
        [pred['x1'], pred['y1'], pred['x2'], pred['y2']])
sahi_preds_by_img = {k: np.array(v, dtype=np.float32) if v else np.zeros((0,4))
                     for k, v in sahi_preds_by_img.items()}

# ── Compute recall per bucket ─────────────────────────────────────────────────
bucket_results = []
for label_path in VAL_LABELS_DIR.glob('*.txt'):
    img_path = VAL_IMAGES_DIR / (label_path.stem + '.jpg')
    if not img_path.exists():
        continue
    img        = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2]
    gt_norm, _   = load_label(label_path)
    gt_xyxy      = yolo_to_xyxy(gt_norm, img_w, img_h)
    if len(gt_xyxy) == 0:
        continue

    base_boxes = base_preds_by_img.get(label_path.stem, np.zeros((0,4)))
    sahi_boxes = sahi_preds_by_img.get(label_path.stem, np.zeros((0,4)))

    base_iou = iou_matrix(gt_xyxy, base_boxes).max(axis=1) if len(base_boxes) else np.zeros(len(gt_xyxy))
    sahi_iou = iou_matrix(gt_xyxy, sahi_boxes).max(axis=1) if len(sahi_boxes) else np.zeros(len(gt_xyxy))

    px_sides = np.sqrt(
        (gt_xyxy[:,2]-gt_xyxy[:,0]) * (gt_xyxy[:,3]-gt_xyxy[:,1])
    )  # effective side length

    for gt_i in range(len(gt_xyxy)):
        bucket_results.append({
            'px_side':      float(px_sides[gt_i]),
            'base_hit':     int(base_iou[gt_i] >= IOU_THRESH),
            'sahi_hit':     int(sahi_iou[gt_i] >= IOU_THRESH),
        })

bucket_df = pd.DataFrame(bucket_results)

# ── Aggregate recall per bucket ───────────────────────────────────────────────
bucket_names, base_recalls, sahi_recalls, counts = [], [], [], []
for name, lo, hi in SIZE_BUCKETS:
    sub = bucket_df[(bucket_df['px_side'] >= lo) & (bucket_df['px_side'] < hi)]
    if len(sub) == 0:
        continue
    bucket_names.append(name)
    base_recalls.append(sub['base_hit'].mean())
    sahi_recalls.append(sub['sahi_hit'].mean())
    counts.append(len(sub))
    print(f'  {name:<12} n={len(sub):>6,}  base recall={sub["base_hit"].mean()*100:.1f}%'
          f'  sahi recall={sub["sahi_hit"].mean()*100:.1f}%'
          f'  delta={( sub["sahi_hit"].mean()-sub["base_hit"].mean())*100:+.1f}pp')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x  = np.arange(len(bucket_names))
bw = 0.35
ax.bar(x - bw/2, [r*100 for r in base_recalls], bw, label='Baseline', color='#7f8c8d', alpha=0.85)
ax.bar(x + bw/2, [r*100 for r in sahi_recalls], bw, label='SAHI 512/0.2', color='#2980b9', alpha=0.85)

for i in range(len(bucket_names)):
    delta = (sahi_recalls[i] - base_recalls[i]) * 100
    ax.text(x[i]+bw/2, sahi_recalls[i]*100+0.5, f'{delta:+.1f}pp',
            ha='center', fontsize=9, color='darkgreen' if delta > 0 else 'darkred')
    ax.text(x[i], -5, f'n={counts[i]:,}', ha='center', fontsize=8, color='#555')

ax.set_xticks(x)
ax.set_xticklabels([f'{n}' for n in bucket_names])
ax.set_ylabel('Recall (%)')
ax.set_ylim(-8, 105)
ax.set_title('Recall by Object Size Bucket: Baseline vs SAHI (IoU≥0.5)', fontsize=11)
ax.legend()
ax.axhline(0, color='black', lw=0.5)

plt.tight_layout()
out_path = FIGURES_DIR / 'sahi_small_object_recall.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

---
## 10. Summary

In [ ]:
print('=' * 60)
print('  SAHI Results — YOLOv8s + SAHI @ 512/0.2')
print('=' * 60)
print(f'  mAP@0.5        : {sahi_map50*100:.2f}%  ({delta_map50*100:+.2f}pp vs baseline {BASELINE_MAP50*100:.2f}%)')
print(f'  mAP@0.5:0.95   : {sahi_map5095*100:.2f}%  ({delta_map5095*100:+.2f}pp vs baseline {BASELINE_MAP5095*100:.2f}%)')
print()
print('  Weak-spot improvement:')
for name in ['bicycle', 'people', 'three_wheeler']:
    row = per_class_df[per_class_df['class_name'] == name].iloc[0]
    print(f'    {name:<15} {row["baseline_AP50"]*100:.1f}% → {row["sahi_AP50"]*100:.1f}%'
          f'  ({row["delta_AP50"]*100:+.1f}pp)')
print()
print('  Saved files:')
for f in [
    FIGURES_DIR / 'sahi_visual_comparison.png',
    FIGURES_DIR / 'sahi_per_class_comparison.png',
    FIGURES_DIR / 'sahi_small_object_recall.png',
    ANALYSIS_DIR / 'sahi_metrics.csv',
]:
    print(f'    {f}')
print()
print('  Next: research/04_sahi_experiments.ipynb')
print('  Grid search: slice [320,512,640] × overlap [0.1,0.2,0.3]')
print('=' * 60)